In [1]:
from pathlib import Path

# Tìm thư mục dataset từ vị trí notebook đang chạy.
possible_dirs = [
    Path.cwd() / "dataset_dan_su_dat_dai",
    Path.cwd().parent / "dataset_dan_su_dat_dai"
]

DATA_DIR = next(
    (
        path.resolve()
        for path in possible_dirs
        if (path / "candidates_raw.jsonl").is_file()
    ),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError("Không tìm thấy thư mục dataset.")

# Tìm các lần chạy preprocessing đã lưu.
processed_files = sorted(
    (DATA_DIR / "preprocessing_v01").glob(
        "*/documents_preprocessed.jsonl"
    )
)

print("Thư mục dữ liệu:", DATA_DIR)
print("Số file preprocessing tìm thấy:", len(processed_files))

for path in processed_files:
    print(path)

Thư mục dữ liệu: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai
Số file preprocessing tìm thấy: 1
C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\preprocessing_v01\20261105_152613\documents_preprocessed.jsonl


In [2]:
import json

if not processed_files:
    raise FileNotFoundError(
        "Chưa tìm thấy documents_preprocessed.jsonl. "
        "Kiểm tra lại kết quả bước 1."
    )

# Chọn lần chạy mới nhất theo tên thư mục thời gian.
INPUT_FILE = processed_files[-1]

documents = []

with INPUT_FILE.open("r", encoding="utf-8-sig") as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(
                f"JSON lỗi tại dòng {line_number}"
            ) from error

        documents.append(record)

if not documents:
    raise RuntimeError("File không chứa bản ghi nào.")

print("Đang đọc:", INPUT_FILE)
print("Số bản ghi:", len(documents))

# Xem thông tin một bản, chưa cắt hoặc sửa nội dung.
sample = documents[0]
source = sample["source_record"]
text = sample["preprocessing"]["text_clean"]

print("\nMã nguồn:", source.get("doc_name"))
print("Số bản án theo nguồn:", source.get("official_document_id"))
print("Nhóm rà soát:", sample["preprocessing"].get("review_route"))
print("\n500 ký tự đầu:\n")
print(text[:500])

Đang đọc: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\preprocessing_v01\20261105_152613\documents_preprocessed.jsonl
Số bản ghi: 5000

Mã nguồn: 304
Số bản án theo nguồn: 117/2017/DS-PT
Nhóm rà soát: judgment_candidate_review

500 ký tự đầu:

## Page 1

NHÂN DANH
NƯỚC CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
TÒA ÁN NHÂN DÂN TỈNH TÂY NINH
- Thành phần Hội đồng xét xử phúc thẩm gồm có:
Thẩm phán - Chủ tọa phiên tòa: Ông Mai Văn Triến.
Các Thẩm phán:
1. Bà Võ Thị Hồng Mai;
2. Bà Đặng Thị Đồng.
- Thư ký phiên tòa: Bà Nguyễn Thị Tuyết Minh, Thư ký Tòa án nhân dân
tỉnh Tây Ninh.
- Đại diện Viện kiểm sát nhân dân tỉnh Tây Ninh tham gia phiên tòa: Bà
Đặng Thị Châu, Kiểm sát viên.
Ngày 10 tháng 7 năm 2017, tại trụ sở Tòa án nhân dân t ỉnh Tây Ninh 


In [3]:
from collections import Counter

route_counts = Counter(
    doc["preprocessing"].get("review_route", "unknown")
    for doc in documents
)

print("PHÂN NHÓM DỮ LIỆU")
for group, count in route_counts.items():
    print(f"{group}: {count:,}")

# Giữ toàn bộ dữ liệu; tạo danh sách ưu tiên riêng.
judgment_candidates = [
    doc for doc in documents
    if doc["preprocessing"].get("review_route")
    == "judgment_candidate_review"
]

other_documents = [
    doc for doc in documents
    if doc["preprocessing"].get("review_route")
    != "judgment_candidate_review"
]

assert (
    len(judgment_candidates) + len(other_documents)
    == len(documents)
)

print("\nBản án ưu tiên tách:", len(judgment_candidates))
print("Tài liệu giữ riêng để rà:", len(other_documents))
print("Tổng vẫn giữ:", len(documents))

PHÂN NHÓM DỮ LIỆU
judgment_candidate_review: 2,944
procedural_decision_review: 1,641
quality_or_metadata_review: 415

Bản án ưu tiên tách: 2944
Tài liệu giữ riêng để rà: 2056
Tổng vẫn giữ: 5000


In [4]:
import re
from collections import Counter

# Nhận diện các tiêu đề đứng riêng trên một dòng.
# Có hỗ trợ dấu # của Markdown và dấu : cuối tiêu đề.
heading_pattern = re.compile(
    r"^[ \t#]*(?:"
    r"NỘI[ \t]+DUNG[ \t]+VỤ[ \t]+ÁN"
    r"|NHẬN[ \t]+ĐỊNH[ \t]+CỦA[ \t]+T[ÒO]A[ \t]+ÁN"
    r"|QUYẾT[ \t]+ĐỊNH"
    r")[ \t:：]*$",
    flags=re.IGNORECASE | re.MULTILINE
)

heading_inventory = []

for doc in documents:
    source = doc["source_record"]
    text = doc["preprocessing"]["text_clean"]

    headings = []

    for match in heading_pattern.finditer(text):
        headings.append({
            "heading": match.group().strip(),
            "start": match.start(),
            "end": match.end()
        })

    heading_inventory.append({
        "doc_name": str(source.get("doc_name") or ""),
        "review_route": doc["preprocessing"].get("review_route"),
        "headings": headings
    })

counts = Counter(
    "found" if row["headings"] else "not_found"
    for row in heading_inventory
)

print("Đã kiểm tra:", len(heading_inventory))
print("Có tiêu đề nhận diện được:", counts["found"])
print("Chưa nhận diện được tiêu đề:", counts["not_found"])

# Xem một bản án ưu tiên có tiêu đề.
example = next(
    (
        row for row in heading_inventory
        if row["review_route"] == "judgment_candidate_review"
        and row["headings"]
    ),
    None
)

if example:
    print("\nVí dụ mã nguồn:", example["doc_name"])

    for heading in example["headings"]:
        print(
            heading["heading"],
            "→ vị trí ký tự:",
            heading["start"]
        )

Đã kiểm tra: 5000
Có tiêu đề nhận diện được: 4923
Chưa nhận diện được tiêu đề: 77

Ví dụ mã nguồn: 304
NỘI DUNG VỤ ÁN: → vị trí ký tự: 4066
NHẬN ĐỊNH CỦA TÒA ÁN: → vị trí ký tự: 13703


In [5]:
sectioned_documents = []

for doc, inventory in zip(documents, heading_inventory):
    source = doc["source_record"]
    text = doc["preprocessing"]["text_clean"]
    headings = inventory["headings"]

    sections = []

    def add_section(section_type, start, end):
        if start < end:
            sections.append({
                "section_type": section_type,
                "start": start,
                "end": end,
                "text": text[start:end]
            })

    if not headings:
        add_section("unsegmented", 0, len(text))

    else:
        # Giữ phần trước tiêu đề đầu tiên.
        add_section("opening", 0, headings[0]["start"])

        for index, heading in enumerate(headings):
            start = heading["start"]

            end = (
                headings[index + 1]["start"]
                if index + 1 < len(headings)
                else len(text)
            )

            title = heading["heading"].strip("# \t:：").upper()

            if title.startswith("NỘI"):
                section_type = "case_content_candidate"
            elif title.startswith("NHẬN"):
                section_type = "court_reasoning_candidate"
            else:
                section_type = "decision_candidate"

            add_section(section_type, start, end)

    # Bảo đảm không mất hoặc thay đổi ký tự khi phân đoạn.
    reconstructed = "".join(section["text"] for section in sections)

    assert reconstructed == text, (
        f"Phân đoạn làm mất nội dung ở mã {source.get('doc_name')}"
    )

    sectioned_documents.append({
        "doc_name": str(source.get("doc_name") or ""),
        "review_route": inventory["review_route"],
        "sections": sections,
        "annotation_status": "draft_not_verified"
    })

print("Đã phân đoạn:", len(sectioned_documents))
print("Đã kiểm tra: các khối ghép lại khớp toàn bộ văn bản đầu vào.")

# Thống kê số khối, không phải số bản án.
section_counts = Counter(
    section["section_type"]
    for doc in sectioned_documents
    for section in doc["sections"]
)

for section_type, count in section_counts.items():
    print(f"{section_type}: {count:,} khối")

Đã phân đoạn: 5000
Đã kiểm tra: các khối ghép lại khớp toàn bộ văn bản đầu vào.
opening: 4,923 khối
case_content_candidate: 3,192 khối
court_reasoning_candidate: 3,117 khối
decision_candidate: 6,584 khối
unsegmented: 77 khối


In [6]:
# Các dấu hiệu để tìm đoạn cần kiểm tra.
risk_patterns = {
    "first_instance_result_reference": re.compile(
        r"bản\s+án.{0,100}sơ\s+thẩm"
        r"|t[oò]a\s+án\s+cấp\s+sơ\s+thẩm.{0,60}"
        r"(?:tuyên|quyết\s+định|chấp\s+nhận|không\s+chấp\s+nhận)",
        flags=re.IGNORECASE | re.DOTALL
    ),
    "appeal_information": re.compile(
        r"kháng\s+cáo|kháng\s+nghị"
        r"|phiên\s+t[oò]a\s+phúc\s+thẩm"
        r"|tại\s+cấp\s+phúc\s+thẩm",
        flags=re.IGNORECASE
    ),
    "possible_outcome_statement": re.compile(
        r"chấp\s+nhận\s+(?:toàn\s+bộ\s+|một\s+phần\s+)?"
        r"yêu\s+cầu\s+khởi\s+kiện"
        r"|không\s+chấp\s+nhận\s+yêu\s+cầu"
        r"|bác\s+yêu\s+cầu",
        flags=re.IGNORECASE
    )
}

risk_flags = []

for doc in sectioned_documents:
    for section_index, section in enumerate(doc["sections"]):
        # Kiểm tra khối nội dung và khối chưa phân đoạn.
        if section["section_type"] not in {
            "case_content_candidate",
            "unsegmented"
        }:
            continue

        section_text = section["text"]

        for risk_type, pattern in risk_patterns.items():
            for match in pattern.finditer(section_text):
                # Lấy thêm ngữ cảnh hai bên để đọc.
                local_start = max(0, match.start() - 150)
                local_end = min(
                    len(section_text),
                    match.end() + 250
                )

                risk_flags.append({
                    "doc_name": doc["doc_name"],
                    "section_index": section_index,
                    "section_type": section["section_type"],
                    "risk_type": risk_type,
                    "match_start": section["start"] + match.start(),
                    "match_end": section["start"] + match.end(),
                    "matched_text": match.group(),
                    "context": section_text[local_start:local_end],
                    "review_status": "pending"
                })

print("Số vị trí bị gắn cờ:", len(risk_flags))
print(
    "Số văn bản có cảnh báo:",
    len({row["doc_name"] for row in risk_flags})
)

for risk_type, count in Counter(
    row["risk_type"] for row in risk_flags
).items():
    print(f"{risk_type}: {count:,}")

# Xem 5 cảnh báo đầu tiên.
for row in risk_flags[:5]:
    print("\n" + "=" * 60)
    print("Mã:", row["doc_name"])
    print("Cảnh báo:", row["risk_type"])
    print(row["context"])

Số vị trí bị gắn cờ: 33112
Số văn bản có cảnh báo: 2959
first_instance_result_reference: 10,272
appeal_information: 16,447
possible_outcome_statement: 6,393

Mã: 304
Cảnh báo: first_instance_result_reference
Cao Thị N2 đứng tên.
Anh Mang Hiếu N1 phải chịu 8.500.000 đồng tiền chi phí đo đ ạc và thẩm
định giá tài sản. Ghi nhận: Anh N1 đã nộp xong.
Ngoài ra, bản án sơ thẩm còn tuyên về nghĩa vụ chậm trả, án phí; quyền
kháng cáo; quyền, nghĩa vụ của các đương sự theo Luật Thi hành án dân sự.
Ngày 24 tháng 01 năm 2017, anh Mang Văn M có đơn kháng cáo v ới nội
dung yêu cầu xem xét lại toàn bộ bản án sơ thẩm.
Ngày 04 thán

Mã: 304
Cảnh báo: first_instance_result_reference
ụ của các đương sự theo Luật Thi hành án dân sự.
Ngày 24 tháng 01 năm 2017, anh Mang Văn M có đơn kháng cáo v ới nội
dung yêu cầu xem xét lại toàn bộ bản án sơ thẩm.
Ngày 04 tháng 02 năm 2017, anh Mang Ngọc D1 có đơn kháng cáo v ới
nội dung: Xem xét, đánh giá l ại toàn b ộ chứng cứ và xem xét l ại toàn b ộ quá
trình tố tụng,

In [7]:
import json
import pandas as pd
from datetime import datetime

BUILD_DIR = (
    DATA_DIR
    / "dataset_build_v01"
    / datetime.now().strftime("%Y%m%d_%H%M%S")
)
BUILD_DIR.mkdir(parents=True, exist_ok=False)

# Lưu các khối văn bản, chưa xóa phần nào.
with (BUILD_DIR / "document_sections_draft.jsonl").open(
    "w", encoding="utf-8"
) as file:
    for doc in sectioned_documents:
        file.write(json.dumps(doc, ensure_ascii=False) + "\n")

# Lưu từng vị trí cảnh báo.
risk_columns = [
    "doc_name", "section_index", "section_type", "risk_type",
    "match_start", "match_end", "matched_text", "context",
    "review_status"
]

risk_df = pd.DataFrame(risk_flags, columns=risk_columns)

risk_df.to_csv(
    BUILD_DIR / "risk_flags.csv",
    index=False,
    encoding="utf-8-sig"
)

# Tổng hợp số cảnh báo theo từng văn bản.
if not risk_df.empty:
    risk_summary = pd.crosstab(
        risk_df["doc_name"],
        risk_df["risk_type"]
    )
    risk_summary["total_flags"] = risk_summary.sum(axis=1)
    risk_summary = risk_summary.reset_index()
else:
    risk_summary = pd.DataFrame(
        columns=["doc_name", "total_flags"]
    )

risk_summary.to_csv(
    BUILD_DIR / "risk_summary_by_document.csv",
    index=False,
    encoding="utf-8-sig"
)

# Ghi rõ file nguồn để đối chiếu vị trí ký tự.
manifest = {
    "input_file": str(INPUT_FILE),
    "document_count": len(sectioned_documents),
    "risk_position_count": len(risk_flags),
    "status": "draft_not_verified",
    "offset_reference": "preprocessing.text_clean",
    "note": "Chưa tạo facts hoặc nhãn; chưa loại nội dung."
}

(BUILD_DIR / "build_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Đã lưu tại:", BUILD_DIR)
print("Số văn bản:", len(sectioned_documents))
print("Số vị trí cảnh báo:", len(risk_df))

Đã lưu tại: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\dataset_build_v01\20261105_193940
Số văn bản: 5000
Số vị trí cảnh báo: 33112


In [8]:
from collections import defaultdict

# Gom cảnh báo theo văn bản và khối.
risks_by_section = defaultdict(list)

for flag in risk_flags:
    key = (flag["doc_name"], flag["section_index"])
    risks_by_section[key].append(flag)

paragraph_rows = []

for doc in sectioned_documents:
    paragraph_number = 0

    for section_index, section in enumerate(doc["sections"]):
        section_text = section["text"]
        section_risks = risks_by_section.get(
            (doc["doc_name"], section_index), []
        )

        # Tách theo dòng trắng, giữ vị trí trong text_clean.
        for match in re.finditer(
            r"\S[\s\S]*?(?=\n[ \t]*\n|\Z)",
            section_text
        ):
            paragraph_number += 1

            start = section["start"] + match.start()
            end = section["start"] + match.end()

            risk_types = sorted({
                flag["risk_type"]
                for flag in section_risks
                if flag["match_start"] < end
                and flag["match_end"] > start
            })

            paragraph_rows.append({
                "doc_name": doc["doc_name"],
                "paragraph_id": (
                    f"{doc['doc_name']}-P{paragraph_number:04d}"
                ),
                "section_type": section["section_type"],
                "start": start,
                "end": end,
                "text": match.group(),
                "risk_types": "; ".join(risk_types),

                # Để trống cho bước phân loại nội dung.
                "content_role": "",
                "temporal_status": "",
                "use_as_input": "",
                "review_notes": ""
            })

paragraph_df = pd.DataFrame(paragraph_rows)

paragraph_df.to_csv(
    BUILD_DIR / "paragraphs_review.csv",
    index=False,
    encoding="utf-8-sig"
)

with (BUILD_DIR / "paragraphs_draft.jsonl").open(
    "w", encoding="utf-8"
) as file:
    for row in paragraph_rows:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Tổng số đoạn:", len(paragraph_df))
print(
    "Số đoạn có cảnh báo:",
    int(paragraph_df["risk_types"].ne("").sum())
)

display(
    paragraph_df[
        ["paragraph_id", "section_type", "risk_types", "text"]
    ].head(5)
)

print("Đã lưu tại:", BUILD_DIR)

Tổng số đoạn: 136348
Số đoạn có cảnh báo: 7852


,paragraph_id,section_type,risk_types,text
0,304-P0001,opening,,## Page 1
1,304-P0002,opening,,NHÂN DANH\nNƯỚC CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT...
2,304-P0003,opening,,Bản án số: 117/2017/DS-PT\nNgày 10-7-2017\nV/v...
3,304-P0004,opening,,CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM\n ...
4,304-P0005,opening,,## Page 2


Đã lưu tại: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\dataset_build_v01\20261105_193940


In [9]:
def suggest_action(row):
    section_type = row["section_type"]
    has_risk = bool(row["risk_types"])

    if section_type == "opening":
        return (
            "opening_or_metadata",
            "not_assessed",
            "review"
        )

    if section_type == "court_reasoning_candidate":
        return (
            "court_reasoning_candidate",
            "not_assessed",
            "exclude_from_prediction_input_candidate"
        )

    if section_type == "decision_candidate":
        return (
            "court_decision_candidate",
            "not_assessed",
            "exclude_from_prediction_input_candidate"
        )

    if has_risk:
        return (
            "mixed_content_risk",
            "uncertain",
            "review"
        )

    return (
        "case_content_candidate",
        "uncertain",
        "review"
    )


suggestions = [
    suggest_action(row)
    for row in paragraph_rows
]

paragraph_df["suggested_role"] = [
    item[0] for item in suggestions
]

paragraph_df["suggested_temporal_status"] = [
    item[1] for item in suggestions
]

paragraph_df["suggested_action"] = [
    item[2] for item in suggestions
]

# Lưu sang file mới, không ghi đè bảng rà soát trước.
suggestion_file = BUILD_DIR / "paragraphs_with_suggestions.csv"

paragraph_df.to_csv(
    suggestion_file,
    index=False,
    encoding="utf-8-sig"
)

print("PHÂN LUỒNG ĐOẠN:")
display(
    paragraph_df["suggested_action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="count")
)

print("Đã lưu:", suggestion_file)

PHÂN LUỒNG ĐOẠN:


,action,count
0,review,72461
1,exclude_from_prediction_input_candidate,63887


Đã lưu: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\dataset_build_v01\20261105_193940\paragraphs_with_suggestions.csv


In [10]:
def classify_paragraph(row):
    section_type = row["section_type"]
    text = row["text"]

    # Chuẩn hóa khoảng trắng, không sửa nguyên văn.
    searchable = re.sub(r"\s+", " ", text).strip().lower()

    role = "unclassified"
    temporal = "uncertain"
    action = "review"
    reasons = []

    # Ưu tiên đánh dấu các khối do tiêu đề xác định.
    if section_type == "opening":
        role = "opening_or_metadata"
        reasons.append("opening_section")

    elif section_type == "court_reasoning_candidate":
        role = "court_reasoning_candidate"
        action = "exclude_from_input_candidate"
        reasons.append("reasoning_heading")

    elif section_type == "decision_candidate":
        role = "court_decision_candidate"
        action = "exclude_from_input_candidate"
        reasons.append("decision_heading")

    else:
        # Những cụm này chỉ là dấu hiệu, không xác minh thời điểm.
        if re.search(
            r"tại\s+phiên\s+t[oò]a\s+phúc\s+thẩm"
            r"|tại\s+cấp\s+phúc\s+thẩm",
            searchable
        ):
            role = "appeal_stage_candidate"
            temporal = "post_first_instance_candidate"
            reasons.append("appeal_stage_phrase")

        elif re.search(
            r"kháng\s+cáo|kháng\s+nghị",
            searchable
        ):
            role = "appeal_related_candidate"
            reasons.append("appeal_related_phrase")

        elif re.search(
            r"(?:bản\s+án|t[oò]a\s+án).{0,120}sơ\s+thẩm"
            r".{0,120}(?:tuyên|quyết\s+định|xử)",
            searchable
        ):
            role = "first_instance_result_reference_candidate"
            reasons.append("first_instance_result_phrase")

        else:
            # Một đoạn có thể chứa nhiều vai trò.
            roles = []

            role_patterns = {
                "plaintiff_statement_candidate": (
                    r"nguyên\s+đơn.{0,180}"
                    r"(?:trình\s+bày|khai|cho\s+rằng)"
                ),
                "defendant_statement_candidate": (
                    r"bị\s+đơn.{0,180}"
                    r"(?:trình\s+bày|khai|cho\s+rằng)"
                ),
                "related_party_statement_candidate": (
                    r"người\s+có\s+quyền\s+lợi.{0,200}"
                    r"(?:trình\s+bày|khai)"
                ),
                "witness_statement_candidate": (
                    r"người\s+làm\s+chứng.{0,180}"
                    r"(?:trình\s+bày|khai)"
                ),
                "evidence_description_candidate": (
                    r"biên\s+bản.{0,60}(?:thẩm\s+định|đo\s+đạc)"
                    r"|kết\s+luận\s+giám\s+định"
                    r"|trích\s+đo\s+địa\s+chính"
                ),
                "request_mention_candidate": (
                    r"yêu\s+cầu\s+(?:t[oò]a|buộc|công\s+nhận|"
                    r"tuyên|hủy|huỷ|trả|tháo)"
                )
            }

            for name, pattern in role_patterns.items():
                if re.search(pattern, searchable):
                    roles.append(name)

            if roles:
                role = "; ".join(roles)
                reasons.append("keyword_role_hint")
            else:
                role = "case_content_candidate"

    if row["risk_types"]:
        reasons.append("risk_flag_present")

    return {
        "suggested_role": role,
        "suggested_temporal_status": temporal,
        "suggested_action": action,
        "suggestion_reasons": "; ".join(reasons)
    }


suggestions = pd.DataFrame([
    classify_paragraph(row)
    for row in paragraph_df.to_dict("records")
])

# Gán riêng từng cột; chạy lại ô không tạo cột trùng.
for column in suggestions.columns:
    paragraph_df[column] = suggestions[column].to_numpy()

print("Số đoạn:", len(paragraph_df))

display(
    paragraph_df["suggested_action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="count")
)

display(
    paragraph_df[
        [
            "paragraph_id",
            "suggested_role",
            "suggested_temporal_status"
        ]
    ].head(10)
)

Số đoạn: 136348


,action,count
0,review,72461
1,exclude_from_input_candidate,63887


,paragraph_id,suggested_role,suggested_temporal_status
0,304-P0001,opening_or_metadata,uncertain
1,304-P0002,opening_or_metadata,uncertain
2,304-P0003,opening_or_metadata,uncertain
3,304-P0004,opening_or_metadata,uncertain
4,304-P0005,opening_or_metadata,uncertain
5,304-P0006,opening_or_metadata,uncertain
6,304-P0007,opening_or_metadata,uncertain
7,304-P0008,opening_or_metadata,uncertain
8,304-P0009,opening_or_metadata,uncertain
9,304-P0010,plaintiff_statement_candidate; request_mention...,uncertain


In [11]:
# Metadata nguồn được giữ riêng với các gợi ý.
source_lookup = {
    str(doc["source_record"].get("doc_name") or ""): doc
    for doc in documents
}

document_rows = []

for sectioned in sectioned_documents:
    doc_id = sectioned["doc_name"]
    original = source_lookup[doc_id]
    source = original["source_record"]
    preprocessing = original["preprocessing"]

    paragraphs = paragraph_df[
        paragraph_df["doc_name"].eq(doc_id)
    ]

    sections = sectioned["sections"]
    section_types = {
        section["section_type"]
        for section in sections
    }

    document_rows.append({
        "doc_name": doc_id,
        "official_id_source": source.get("official_document_id"),
        "court_source": source.get("court"),
        "issued_date_source": source.get("issued_date"),
        "instance_source": source.get("instance_level"),
        "instance_guess": preprocessing.get("instance_guess"),
        "document_type_guess": preprocessing.get(
            "document_type_guess"
        ),
        "review_route": preprocessing.get("review_route"),
        "candidate_groups": "; ".join(
            (original.get("screening") or {}).get(
                "candidate_groups", []
            )
        ),
        "paragraph_count": len(paragraphs),
        "paragraphs_with_risk": int(
            paragraphs["risk_types"].ne("").sum()
        ),
        "has_content_section": (
            "case_content_candidate" in section_types
        ),
        "has_reasoning_section": (
            "court_reasoning_candidate" in section_types
        ),
        "has_decision_section": (
            "decision_candidate" in section_types
        ),
        "web_url": source.get("web_url"),
        "pdf_url": source.get("pdf_url"),

        # Điền sau khi người rà soát đọc nguồn.
        "verified_official_id": "",
        "verified_court": "",
        "verified_date": "",
        "verified_instance": "",
        "case_group_id": "",
        "eligible_for_prediction": "",
        "review_notes": ""
    })

document_inventory = pd.DataFrame(document_rows)

# Tất cả các đoạn vẫn cần kiểm tra trước khi thành đầu vào chuẩn.
paragraph_review = paragraph_df.copy()

# Bảng yêu cầu và kết quả để điền sau.
# Không tự gán một yêu cầu cho mỗi bản.
claim_columns = [
    "sample_id",
    "doc_name",
    "case_group_id",
    "requesting_party",
    "request_text",
    "request_paragraph_ids",
    "fact_paragraph_ids",
    "first_instance_outcome_quote",
    "outcome_source_paragraph_ids",
    "label",
    "temporal_review_status",
    "annotation_status",
    "review_notes"
]

claim_annotations = pd.DataFrame(columns=claim_columns)

print("Danh mục văn bản:", len(document_inventory))
print("Đoạn đang chờ rà:", len(paragraph_review))
print("Mẫu yêu cầu đã xác minh:", len(claim_annotations))

Danh mục văn bản: 5000
Đoạn đang chờ rà: 136348
Mẫu yêu cầu đã xác minh: 0


In [12]:
from datetime import datetime, timezone

# Kiểm tra đoạn trích và vị trí ký tự.
for row in paragraph_df.to_dict("records"):
    text = source_lookup[row["doc_name"]][
        "preprocessing"
    ]["text_clean"]

    assert text[row["start"]:row["end"]] == row["text"], (
        f"Sai vị trí nguồn: {row['paragraph_id']}"
    )

# Tạo thư mục mới, không ghi đè các lần rà trước.
EXPORT_DIR = (
    BUILD_DIR
    / ("export_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
)
EXPORT_DIR.mkdir(parents=True, exist_ok=False)


def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as file:
        for row in rows:
            file.write(
                json.dumps(row, ensure_ascii=False) + "\n"
            )


def write_csv(path, table):
    table.to_csv(
        path,
        index=False,
        encoding="utf-8-sig"
    )


# 1. Văn bản đầy đủ, giữ cả record nguồn.
write_jsonl(
    EXPORT_DIR / "documents.jsonl",
    documents
)

# 2. Các khối theo tiêu đề.
write_jsonl(
    EXPORT_DIR / "document_sections_draft.jsonl",
    sectioned_documents
)

# 3. Từng đoạn và các gợi ý.
write_jsonl(
    EXPORT_DIR / "paragraphs_draft.jsonl",
    paragraph_df.to_dict("records")
)

write_csv(
    EXPORT_DIR / "paragraphs_review.csv",
    paragraph_review
)

write_csv(
    EXPORT_DIR / "document_inventory.csv",
    document_inventory
)

write_csv(
    EXPORT_DIR / "claim_annotations_template.csv",
    claim_annotations
)

# 4. Cảnh báo.
write_csv(
    EXPORT_DIR / "risk_flags.csv",
    pd.DataFrame(risk_flags)
)

# 5. Mẫu đoạn có gợi ý yêu cầu.
request_candidates = paragraph_df[
    paragraph_df["suggested_role"].str.contains(
        "request_mention_candidate",
        regex=False
    )
]

write_csv(
    EXPORT_DIR / "request_mentions_review.csv",
    request_candidates
)

# 6. Mẫu đoạn có thể thuật lại kết quả sơ thẩm.
outcome_candidates = paragraph_df[
    paragraph_df["suggested_role"].eq(
        "first_instance_result_reference_candidate"
    )
]

write_csv(
    EXPORT_DIR / "first_instance_result_mentions_review.csv",
    outcome_candidates
)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "input_file": str(INPUT_FILE.resolve()),
    "documents": len(documents),
    "paragraphs": len(paragraph_df),
    "risk_positions": len(risk_flags),
    "request_mention_paragraphs": len(request_candidates),
    "first_instance_result_mention_paragraphs": len(
        outcome_candidates
    ),
    "verified_prediction_samples": 0,
    "status": "draft_not_verified",
    "offset_reference": "preprocessing.text_clean",
    "notes": [
        "Không xóa toàn văn hoặc ghi đè dữ liệu gốc.",
        "Gợi ý vai trò và thời điểm dựa trên quy tắc.",
        "Chưa xác minh mọi quan hệ trùng vụ.",
        "Chưa tạo nhãn dự đoán hoặc chia train/test."
    ]
}

(EXPORT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

readme = """
# Dataset phân đoạn nháp

## Nội dung
- documents.jsonl: giữ toàn bộ bản ghi và văn bản nguồn.
- document_sections_draft.jsonl: khối theo tiêu đề.
- paragraphs_draft.jsonl: đoạn nguyên văn, vị trí và gợi ý.
- document_inventory.csv: danh mục văn bản.
- paragraphs_review.csv: bảng rà từng đoạn.
- request_mentions_review.csv: đoạn có nhắc yêu cầu.
- first_instance_result_mentions_review.csv: đoạn có thể thuật lại kết quả sơ thẩm.
- claim_annotations_template.csv: mẫu bảng gán nhãn từng yêu cầu.
- risk_flags.csv: các vị trí có cảnh báo.

## Cách rà
1. Kiểm tra số hiệu, ngày, tòa và cấp xét xử từ văn bản.
2. Xác định từng yêu cầu và bên đưa ra yêu cầu.
3. Chọn đoạn dữ kiện có nguồn, xác định thời điểm.
4. Đối chiếu yêu cầu với kết quả sơ thẩm.
5. Lưu nhãn và đoạn kết quả riêng với đầu vào.

## Giới hạn
Các gợi ý tự động chưa phải xác nhận của người rà soát.
Không dùng nguyên khối nội dung làm facts.
Không coi yêu cầu của một bên là kết quả của tòa.
Không dùng quyết định phúc thẩm thay nhãn sơ thẩm.
Chưa có mẫu prediction được xác minh trong gói này.
"""

(EXPORT_DIR / "README.md").write_text(
    readme.strip(),
    encoding="utf-8"
)

print("Kiểm tra vị trí nguồn: ĐẠT")
print("Đã xuất tại:", EXPORT_DIR)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

Kiểm tra vị trí nguồn: ĐẠT
Đã xuất tại: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\dataset_build_v01\20261105_193940\export_20261105_195911_967237
{
  "created_at_utc": "2026-11-05T12:59:30.872123+00:00",
  "input_file": "C:\\Users\\LENOVO\\OneDrive\\Desktop\\L-GPT\\dataset_dan_su_dat_dai\\preprocessing_v01\\20261105_152613\\documents_preprocessed.jsonl",
  "documents": 5000,
  "paragraphs": 136348,
  "risk_positions": 33112,
  "request_mention_paragraphs": 3544,
  "first_instance_result_mention_paragraphs": 1122,
  "verified_prediction_samples": 0,
  "status": "draft_not_verified",
  "offset_reference": "preprocessing.text_clean",
  "notes": [
    "Không xóa toàn văn hoặc ghi đè dữ liệu gốc.",
    "Gợi ý vai trò và thời điểm dựa trên quy tắc.",
    "Chưa xác minh mọi quan hệ trùng vụ.",
    "Chưa tạo nhãn dự đoán hoặc chia train/test."
  ]
}


In [14]:
import shutil

ZIP_PATH = shutil.make_archive(
    str(EXPORT_DIR),
    "zip",
    root_dir=EXPORT_DIR
)

print("ZIP:", ZIP_PATH)

ZIP: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\dataset_build_v01\20261105_193940\export_20261105_195911_967237.zip


In [15]:
from collections import defaultdict
from pathlib import Path
import json
import re

# Dùng kết quả các ô đã chạy.
if "EXPORT_DIR" not in globals():
    raise RuntimeError(
        "Chưa có EXPORT_DIR. Chạy lại ô xuất dữ liệu trước."
    )

EXPORT_DIR = Path(EXPORT_DIR)

paragraph_file = EXPORT_DIR / "paragraphs_draft.jsonl"

if not paragraph_file.is_file():
    raise FileNotFoundError(paragraph_file)

DOSSIER_DIR = EXPORT_DIR / "review_dossiers"
DOSSIER_DIR.mkdir(exist_ok=True)

# Gom đoạn theo mã nguồn.
paragraphs_by_document = defaultdict(list)

with paragraph_file.open("r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            paragraph = json.loads(line)
            paragraphs_by_document[
                paragraph["doc_name"]
            ].append(paragraph)

# Lấy metadata từ documents đang có trong notebook.
metadata_lookup = {
    str(doc["source_record"].get("doc_name") or ""):
        doc["source_record"]
    for doc in documents
}

for doc_id, paragraphs in paragraphs_by_document.items():
    paragraphs.sort(key=lambda item: item["start"])
    source = metadata_lookup.get(doc_id, {})

    safe_id = re.sub(r"[^a-zA-Z0-9_-]", "_", doc_id)

    lines = [
        f"# Hồ sơ rà soát: {doc_id}",
        "",
        f"- Số hiệu nguồn: {source.get('official_document_id')}",
        f"- Tòa nguồn: {source.get('court')}",
        f"- Ngày nguồn: {source.get('issued_date')}",
        f"- Cấp xét xử nguồn: {source.get('instance_level')}",
        f"- Trang nguồn: {source.get('web_url')}",
        "",
        "Thông tin trên chưa được xác minh thủ công.",
        ""
    ]

    for paragraph in paragraphs:
        lines.extend([
            f"## {paragraph['paragraph_id']}",
            "",
            f"- Khối: {paragraph['section_type']}",
            f"- Vai trò gợi ý: {paragraph.get('suggested_role', '')}",
            f"- Cảnh báo: {paragraph.get('risk_types') or 'Không có cờ'}",
            f"- Vị trí: {paragraph['start']}–{paragraph['end']}",
            "",
            paragraph["text"],
            ""
        ])

    (DOSSIER_DIR / f"{safe_id}.md").write_text(
        "\n".join(lines),
        encoding="utf-8"
    )

print("Số hồ sơ đã xuất:", len(paragraphs_by_document))
print("Thư mục:", DOSSIER_DIR.resolve())

Số hồ sơ đã xuất: 5000
Thư mục: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\dataset_build_v01\20261105_193940\export_20261105_195911_967237\review_dossiers


In [16]:
ANNOTATION_DIR = EXPORT_DIR / "claim_annotations"
ANNOTATION_DIR.mkdir(exist_ok=True)

created = 0
existing = 0

for doc_id, paragraphs in paragraphs_by_document.items():
    safe_id = re.sub(r"[^a-zA-Z0-9_-]", "_", doc_id)
    path = ANNOTATION_DIR / f"{safe_id}.json"

    # Không ghi đè hồ sơ đã chỉnh sửa.
    if path.exists():
        existing += 1
        continue

    source = metadata_lookup.get(doc_id, {})

    annotation = {
        "doc_name": doc_id,
        "case_group_id": None,
        "official_id_source": source.get("official_document_id"),
        "instance_source": source.get("instance_level"),
        "review_status": "not_reviewed",
        "eligible": None,
        "review_notes": "",
        "claims": [],
        "claim_template": {
            "sample_id": f"{doc_id}-C01",
            "requesting_party": "",
            "request_text": "",
            "request_paragraph_ids": [],
            "fact_paragraph_ids": [],
            "first_instance_outcome_quote": "",
            "outcome_paragraph_ids": [],
            "label": None,
            "temporal_review_status": "uncertain",
            "annotation_status": "draft",
            "review_notes": ""
        }
    }

    path.write_text(
        json.dumps(annotation, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    created += 1

print("Hồ sơ mới tạo:", created)
print("Hồ sơ có sẵn, giữ nguyên:", existing)
print("Thư mục:", ANNOTATION_DIR.resolve())

Hồ sơ mới tạo: 5000
Hồ sơ có sẵn, giữ nguyên: 0
Thư mục: C:\Users\LENOVO\OneDrive\Desktop\L-GPT\dataset_dan_su_dat_dai\dataset_build_v01\20261105_193940\export_20261105_195911_967237\claim_annotations
